전처리 함수들을 만들고, 결과를 저장할 수 있는 파이프라인을 완성

1  import

2  경로 설정

3  coco 읽기

4  bbox 통계

5  랜덤 이미지 선택

6  저장 폴더

7  이미지 목록 (preview only — img 덮어쓰지 않음)

8  apply_clahe()

9  apply_gamma()

10 apply_frangi()

11 apply_unsharp()

12 apply_tophat() / apply_blackhat()

13 crop_bbox()

14 visualize_preprocessing() 함수

15 visualize_preprocessing() 실행

16 CLAHE clipLimit / tileGrid / 2D 그리드 튜닝

17 Gamma 튜닝

18 Frangi / Unsharp / Black-hat / 조합 튜닝

19 최종 파라미터 반영

20 BATCH Preprocessing


In [1]:
# libarary
from pathlib import Path
import os
import json
import random

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [2]:
# 프로젝트 경로
# cwd가 .../ARCADE/notebooks 일 때 PROJECT_DIR = ARCADE

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"

ARCADE_DIR = DATA_DIR / "arcade" / "arcade"

RAW_DIR = ARCADE_DIR / "stenosis" / "train" / "images"

TRAIN_JSON = (
    ARCADE_DIR
    / "stenosis"
    / "train"
    / "annotations"
    / "train.json"
)

PREPROCESS_DIR = DATA_DIR / "preprocessed"
PREPROCESS_DIR.mkdir(exist_ok=True)

print("cwd           :", Path.cwd())
print("PROJECT_DIR   :", PROJECT_DIR)
print("DATA_DIR      :", DATA_DIR)
print("PREPROCESS_DIR:", PREPROCESS_DIR)
print(TRAIN_JSON)
print("TRAIN_JSON exists:", TRAIN_JSON.exists())
print("RAW_DIR exists   :", RAW_DIR.exists())
print("PREPROCESS exists:", PREPROCESS_DIR.exists())


cwd           : <PROJECT_ROOT>/notebooks
PROJECT_DIR   : <PROJECT_ROOT>
DATA_DIR      : <PROJECT_ROOT>/data
PREPROCESS_DIR: <PROJECT_ROOT>/data/preprocessed
<PROJECT_ROOT>/data/arcade/arcade/stenosis/train/annotations/train.json
TRAIN_JSON exists: True
RAW_DIR exists   : True
PREPROCESS exists: True


In [3]:
# annotation 읽기
with open(TRAIN_JSON, "r") as f:
    coco = json.load(f)

image_info = {
    img["id"]: img
    for img in coco["images"]
}

print(f"Images      : {len(coco['images'])}")
print(f"Annotations : {len(coco['annotations'])}")

Images      : 1000
Annotations : 1625


In [4]:
bbox_width = []
bbox_height = []
bbox_area = []

for ann in coco["annotations"]:

    x, y, w, h = ann["bbox"]

    bbox_width.append(w)
    bbox_height.append(h)
    bbox_area.append(w * h)

print(f"평균 width : {np.mean(bbox_width):.1f}")
print(f"평균 height: {np.mean(bbox_height):.1f}")
print(f"평균 area  : {np.mean(bbox_area):.1f}")

평균 width : 63.7
평균 height: 66.9
평균 area  : 4836.9


In [5]:
# 랜덤 샘플 

ann = random.choice(coco["annotations"])

image_id = ann["image_id"]
bbox = ann["bbox"]

image_name = image_info[image_id]["file_name"]

img = cv2.imread(
    str(RAW_DIR / image_name),
    cv2.IMREAD_GRAYSCALE
)

print(image_name)
print(bbox)

In [6]:
print(image_name)
print(image_id)
print(bbox)

# 같은 이미지의 annotation 개수
anns = [a for a in coco["annotations"] if a["image_id"] == image_id]
print("annotation 개수:", len(anns))

for i, a in enumerate(anns):
    print(i, a["bbox"])

In [7]:
# Preprocess output directories
# Structure: PREPROCESS_DIR / <pipeline> / <split(train|val)> / <filename>.png

folders = [
    '01_clahe',
    '02_gamma',
    '03_frangi',
    '04_clahe_gamma',
    '05_clahe_gamma_frangi',
    '06_gamma_clahe_frangi',
    '07_black_hat',
]

splits = ['train', 'val']

for folder in folders:
    for split in splits:
        d = PREPROCESS_DIR / folder / split
        d.mkdir(parents=True, exist_ok=True)
        print(d, 'exists:', d.exists())

print('folder check done')


<PROJECT_ROOT>/data/preprocessed/01_clahe/train exists: True
<PROJECT_ROOT>/data/preprocessed/01_clahe/val exists: True
<PROJECT_ROOT>/data/preprocessed/02_gamma/train exists: True
<PROJECT_ROOT>/data/preprocessed/02_gamma/val exists: True
<PROJECT_ROOT>/data/preprocessed/03_frangi/train exists: True
<PROJECT_ROOT>/data/preprocessed/03_frangi/val exists: True
<PROJECT_ROOT>/data/preprocessed/04_clahe_gamma/train exists: True
<PROJECT_ROOT>/data/preprocessed/04_clahe_gamma/val exists: True
<PROJECT_ROOT>/data/preprocessed/05_clahe_gamma_frangi/train exists: True
<PROJECT_ROOT>/data/preprocessed/05_clahe_gamma_frangi/val exists: True
<PROJECT_ROOT>/data/preprocessed/06_gamma_clahe_frangi/train exists: True
<PROJECT_ROOT>/data/preprocessed/06_gamma_clahe_frangi/val exists: True
<PROJECT_ROOT>/data/preprocessed/07_black_hat/train exists: True
<PROJECT_ROOT>/data/preprocessed/07_black_hat/val exists: True
folder check done


In [8]:
# 이미지 목록 확인
# 주의: 여기서 img를 덮어쓰면 랜덤 샘플 bbox와 이미지가 어긋남
#       미리보기는 preview 변수만 사용

image_list = sorted(list(RAW_DIR.glob("*.png")))

print(len(image_list))

sample = image_list[0]
preview = cv2.imread(str(sample), cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(6,6))
plt.imshow(preview, cmap="gray")
plt.title(f"preview only: {sample.name}")
plt.axis("off")


1000


(np.float64(-0.5), np.float64(511.5), np.float64(511.5), np.float64(-0.5))

In [9]:
# CLAHE

def apply_clahe(img,
                clip_limit=2.0,
                tile_grid=(16,16)):

    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=tile_grid
    )

    return clahe.apply(img)

In [10]:
# Gamma Correction

# 나중에 gamma=0.8, 1.5, 2.0도 비교

def apply_gamma(img,
                gamma=0.7):

    img = img.astype(np.float32) / 255.0

    img = np.power(img, gamma)

    img = (img * 255).astype(np.uint8)

    return img

In [11]:
from skimage.filters import frangi


def apply_frangi(
    img,
    sigmas=range(1, 4),
    alpha=0.5,
    beta=0.5,
    gamma=15,
    black_ridges=True,
):
    img = img.astype(np.float32) / 255.0

    vessel = frangi(
        img,
        sigmas=sigmas,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
        black_ridges=black_ridges,
    )

    # max가 너무 작으면(거의 미검출) 정규화하면 노이즈만 커짐
    vmax = float(vessel.max())
    if vmax > 1e-6:
        vessel = vessel / vmax
    else:
        vessel = np.zeros_like(vessel)

    vessel = (vessel * 255).astype(np.uint8)
    return vessel


In [12]:
# Frangi smoke test (현재 샘플 img 기준)
img_float = img.astype(np.float32) / 255.0

vessel = frangi(
    img_float,
    sigmas=range(1, 5),
    alpha=0.5,
    beta=0.5,
    gamma=15,
    black_ridges=True,
)

print(vessel.dtype)
print(vessel.min(), vessel.max())
if float(vessel.max()) < 1e-6:
    print("경고: Frangi 응답이 거의 0입니다. 이미지/파라미터를 확인하세요.")


float32
0.0 8.517388e-05


In [13]:
import skimage
print(skimage.__version__)

0.26.0


In [14]:
# unsharp mask

def apply_unsharp(
    img,
    sigma=2.0,
    amount=1.5
):

    blur = cv2.GaussianBlur(img, (0,0), sigma)

    sharpen = cv2.addWeighted(
        img,
        1 + amount,
        blur,
        -amount,
        0
    )

    return np.clip(sharpen, 0, 255).astype(np.uint8)

In [15]:
# top-hat / black-hat
# 밝은 구조 강조: TOPHAT
# 어두운 혈관 강조: BLACKHAT (혈관조영에서 자주 더 맞음)

def apply_tophat(
    img,
    kernel_size=15
):

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (kernel_size, kernel_size)
    )

    return cv2.morphologyEx(
        img,
        cv2.MORPH_TOPHAT,
        kernel
    )


def apply_blackhat(
    img,
    kernel_size=15
):

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (kernel_size, kernel_size)
    )

    return cv2.morphologyEx(
        img,
        cv2.MORPH_BLACKHAT,
        kernel
    )


# 같은 병변을 확대해서 비교하기
전체 이미지 말고, Bounding Box 주변만 crop해서 비교
=> 왜냐하면 우리가 개선하려는 대상은 이미지 전체가 아니라 협착 병변이기 때문
## 병변 주변만 확대하면
- 혈관 경계가 더 선명해졌는지
- 협착 부위 비대가 올라갔는지
- 노이즈가 얼마나 증가했는지

In [16]:
# Crop 함수
# Bbox보다 크게 잘라서 보기 (혈관 앞뒤도 확인)

# padding 시작값은 20 -> 30이나 40으로
def crop_bbox(img, bbox, padding=40):

    x, y, w, h = bbox

    x1 = max(int(x-padding), 0)
    y1 = max(int(y-padding), 0)

    x2 = min(int(x+w+padding), img.shape[1])
    y2 = min(int(y+h+padding), img.shape[0])

    crop = img[y1:y2, x1:x2]

    return crop, x1, y1

In [17]:
# show_preprocessing_comparison 비교 함수

import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_preprocessing(
    img,
    bbox,
    image_name,
    padding=40,

    clip_limit=2.0,
    tile_grid=(8, 8),

    gamma_value=0.8,

    frangi_sigmas=range(1, 6),

    unsharp_sigma=2.0,
    unsharp_amount=1.5,

    tophat_kernel=15,
):

    # -----------------------------
    # Figure
    # -----------------------------
    fig = plt.figure(figsize=(12, 10))
    gs = fig.add_gridspec(
        3,
        3,
        width_ratios=[1.4, 1, 1]
    )

    x, y, w, h = map(int, bbox)

    # -----------------------------
    # preprocessing
    # -----------------------------
    original = img

    clahe = apply_clahe(
        img,
        clip_limit=clip_limit,
        tile_grid=tile_grid
    )

    gamma = apply_gamma(
        img,
        gamma=gamma_value
    )

    # Frangi는 CLAHE 뒤에 적용하는 편이 안정적
    frangi_img = apply_frangi(
        clahe,
        sigmas=frangi_sigmas
    )

    unsharp = apply_unsharp(
        img,
        sigma=unsharp_sigma,
        amount=unsharp_amount
    )

    blackhat = apply_blackhat(
        img,
        kernel_size=tophat_kernel
    )

    images = [
        ("Original", original),
        ("CLAHE", clahe),
        ("Gamma", gamma),
        ("CLAHE+Frangi", frangi_img),
        ("Unsharp", unsharp),
        ("Black-hat", blackhat),
    ]

    # ==========================
    # Left : Original
    # ==========================
    ax0 = fig.add_subplot(gs[:, 0])

    ax0.imshow(original, cmap="gray")

    rect = patches.Rectangle(
        (x, y),
        w,
        h,
        linewidth=2,
        edgecolor="lime",
        facecolor="none"
    )
    ax0.add_patch(rect)

    crop_rect = patches.Rectangle(
        (x - padding, y - padding),
        w + padding * 2,
        h + padding * 2,
        linewidth=2,
        edgecolor="yellow",
        linestyle="--",
        facecolor="none"
    )
    ax0.add_patch(crop_rect)

    ax0.set_title(image_name)
    ax0.axis("off")

    # ==========================
    # Right : Crop Compare
    # ==========================
    positions = [
        gs[0, 1],
        gs[0, 2],
        gs[1, 1],
        gs[1, 2],
        gs[2, 1],
        gs[2, 2],
    ]

    for pos, (title, image) in zip(positions, images):

        crop, x1, y1 = crop_bbox(
            image,
            bbox,
            padding
        )

        ax = fig.add_subplot(pos)
        ax.imshow(crop, cmap="gray")

        crop_rect = patches.Rectangle(
            (x - x1, y - y1),
            w,
            h,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )
        ax.add_patch(crop_rect)

        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [18]:
# ===== Parameter Settings =====

# CLAHE
clip_limit = 2.0
tile_grid = (8, 8)

# Gamma
gamma_value = 0.8

# Frangi
frangi_sigmas = range(1, 5)

# Unsharp Mask
unsharp_sigma = 2.0
unsharp_amount = 1.5

# Top-hat
tophat_kernel = 15

In [19]:
visualize_preprocessing(
    img,
    bbox,
    image_name,
    clip_limit=clip_limit,
    tile_grid=tile_grid,
    gamma_value=gamma_value,
    frangi_sigmas=frangi_sigmas,
    unsharp_sigma=unsharp_sigma,
    unsharp_amount=unsharp_amount,
    tophat_kernel=tophat_kernel,
)

┌─────────────────────────────┬────────────────────────────┬────────────────────────────┐
│                             │ Original Crop              │ CLAHE Crop                 │
│                             ├────────────────────────────┼────────────────────────────┤
│                             │ Gamma Crop                 │ Frangi Crop                │
│ Original                    ├────────────────────────────┼────────────────────────────┤
│ + Bounding Box              │ CLAHE+Gamma Crop           │ CLAHE+Gamma+Frangi Crop    │
│ + Crop Area                 │                            │                            │
└─────────────────────────────┴────────────────────────────┴────────────────────────────┘
왼쪽에는
원본, 빨간 BoudingBox
노랑 Crop 영역

오른쪽 Crop 6개
각 Crop에도 빨간 bounding box

# CLAHE 파라미터 튜닝

In [20]:
# CLAHE파라미터 튜닝 clipLimit만 비교

clip_values = [1.0, 2.0, 3.0, 4.0]

fig, axes = plt.subplots(1, len(clip_values)+1, figsize=(20,5))

# 원본
crop, _, _ = crop_bbox(img, bbox, padding=40)

axes[0].imshow(crop, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

for i, clip in enumerate(clip_values):

    clahe = apply_clahe(
        img,
        clip_limit=clip
    )

    crop_img, x1, y1 = crop_bbox(
        clahe,
        bbox,
        padding=40
    )

    x, y, w, h = bbox

    rect = patches.Rectangle(
        (x-x1, y-y1),
        w,
        h,
        linewidth=2,
        edgecolor="lime",
        facecolor="none"
    )

    axes[i+1].imshow(crop_img, cmap="gray")
    axes[i+1].add_patch(rect)

    axes[i+1].set_title(f"clip={clip}")
    axes[i+1].axis("off")

plt.tight_layout()
plt.show()

### CLAHE tileGrid 튜닝
clipLimit과 같이 봐야 함.
- tile이 작을수록(예: 4x4) 국소 대비↑, 노이즈↑
- tile이 클수록(예: 16x16) 부드러움↑, 작은 혈관 대비↓


In [21]:
# CLAHE tileGrid 튜닝 (clipLimit 고정)

tile_grids = [(4, 4), (8, 8), (16, 16), (32, 32)]

fig, axes = plt.subplots(1, len(tile_grids) + 1, figsize=(22, 5))

crop, _, _ = crop_bbox(img, bbox, padding=40)
axes[0].imshow(crop, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

x, y, w, h = bbox

for i, tg in enumerate(tile_grids):
    clahe = apply_clahe(
        img,
        clip_limit=clip_limit,
        tile_grid=tg,
    )
    crop_img, x1, y1 = crop_bbox(clahe, bbox, padding=40)

    rect = patches.Rectangle(
        (x - x1, y - y1), w, h,
        linewidth=2, edgecolor="lime", facecolor="none"
    )
    axes[i + 1].imshow(crop_img, cmap="gray")
    axes[i + 1].add_patch(rect)
    axes[i + 1].set_title(f"tile={tg}")
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()


### CLAHE clipLimit × tileGrid 그리드 튜닝
두 파라미터를 같이 바꿔서 비교.
눈으로 고른 조합을 아래 셀에서 `clip_limit`, `tile_grid`에 반영하면 됨.


In [22]:
# CLAHE 2D 그리드 튜닝: clipLimit x tileGrid

clip_values = [1.0, 2.0, 3.0, 4.0]
tile_grids = [(4, 4), (8, 8), (16, 16)]

fig, axes = plt.subplots(
    len(tile_grids),
    len(clip_values),
    figsize=(16, 10),
)

x, y, w, h = bbox

for r, tg in enumerate(tile_grids):
    for c, clip in enumerate(clip_values):
        clahe = apply_clahe(img, clip_limit=clip, tile_grid=tg)
        crop_img, x1, y1 = crop_bbox(clahe, bbox, padding=40)

        ax = axes[r, c]
        ax.imshow(crop_img, cmap="gray")
        ax.add_patch(
            patches.Rectangle(
                (x - x1, y - y1), w, h,
                linewidth=1.5, edgecolor="lime", facecolor="none"
            )
        )
        ax.set_title(f"clip={clip}, tile={tg}")
        ax.axis("off")

plt.tight_layout()
plt.show()


판단기준

| 항목      | 좋은 결과      |
| -----    | ---------- |
| 혈관 경계 | 선명         |
| 협착부    | 잘 보임       |
| 노이즈    | 과하지 않음     |
| 배경      | 너무 밝아지지 않음 |


# ============================================
# CLAHE Test Result
#
# clipLimit : 2.0
#
# tileGrid :
#   (4,4)  ← Selected
#   (8,8)
#   (16,16)
#
# Reason
# - Better local vessel contrast
# - Small vessel visibility improved
# - Acceptable noise increase
# ============================================

In [23]:
# gamma 튜닝 코드
gamma_values = [0.6, 0.8, 1.0, 1.2, 1.5]

fig, axes = plt.subplots(1, len(gamma_values)+1, figsize=(20,5))

# Original
crop, _, _ = crop_bbox(img, bbox, padding=40)

axes[0].imshow(crop, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

for i, gamma in enumerate(gamma_values):

    gamma_img = apply_gamma(
        img,
        gamma=gamma
    )

    crop_img, x1, y1 = crop_bbox(
        gamma_img,
        bbox,
        padding=40
    )

    x, y, w, h = bbox

    rect = patches.Rectangle(
        (x-x1, y-y1),
        w,
        h,
        linewidth=2,
        edgecolor="lime",
        facecolor="none"
    )

    axes[i+1].imshow(crop_img, cmap="gray")
    axes[i+1].add_patch(rect)

    axes[i+1].set_title(f"gamma={gamma}")
    axes[i+1].axis("off")

plt.tight_layout()
plt.show()

## 추가 파라미터 튜닝
CLAHE 말고도 아래를 튜닝할 수 있음.
1. Frangi `sigmas` / `black_ridges`
2. Unsharp `sigma` / `amount`
3. Black-hat `kernel_size`
4. (선택) CLAHE → Gamma 조합


In [24]:
# Frangi sigma 튜닝 (CLAHE 적용 후)
# sigma가 작을수록 가는 혈관, 클수록 굵은 혈관에 반응

base = apply_clahe(img, clip_limit=clip_limit, tile_grid=tile_grid)
sigma_sets = [
    range(1, 3),
    range(1, 5),
    range(2, 6),
    range(1, 8),
]

fig, axes = plt.subplots(1, len(sigma_sets) + 1, figsize=(22, 5))

crop, _, _ = crop_bbox(base, bbox, padding=40)
axes[0].imshow(crop, cmap="gray")
axes[0].set_title("CLAHE")
axes[0].axis("off")

x, y, w, h = bbox

for i, sigmas in enumerate(sigma_sets):
    out = apply_frangi(base, sigmas=sigmas, black_ridges=True)
    crop_img, x1, y1 = crop_bbox(out, bbox, padding=40)
    axes[i + 1].imshow(crop_img, cmap="gray")
    axes[i + 1].add_patch(
        patches.Rectangle(
            (x - x1, y - y1), w, h,
            linewidth=2, edgecolor="lime", facecolor="none"
        )
    )
    axes[i + 1].set_title(f"sigmas={list(sigmas)}")
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()


In [25]:
# Frangi black_ridges 비교
# True  : 어두운 혈관(일반적인 조영 혈관)
# False : 밝은 혈관

base = apply_clahe(img, clip_limit=clip_limit, tile_grid=tile_grid)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
x, y, w, h = bbox

variants = [
    ("CLAHE", base),
    ("black_ridges=True", apply_frangi(base, sigmas=frangi_sigmas, black_ridges=True)),
    ("black_ridges=False", apply_frangi(base, sigmas=frangi_sigmas, black_ridges=False)),
]

for ax, (title, im) in zip(axes, variants):
    crop_img, x1, y1 = crop_bbox(im, bbox, padding=40)
    ax.imshow(crop_img, cmap="gray")
    ax.add_patch(
        patches.Rectangle(
            (x - x1, y - y1), w, h,
            linewidth=2, edgecolor="lime", facecolor="none"
        )
    )
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [26]:
# Unsharp 튜닝: sigma / amount

sigmas = [1.0, 2.0, 3.0]
amounts = [0.8, 1.5, 2.5]

fig, axes = plt.subplots(len(sigmas), len(amounts), figsize=(12, 10))
x, y, w, h = bbox

for r, s in enumerate(sigmas):
    for c, a in enumerate(amounts):
        out = apply_unsharp(img, sigma=s, amount=a)
        crop_img, x1, y1 = crop_bbox(out, bbox, padding=40)
        ax = axes[r, c]
        ax.imshow(crop_img, cmap="gray")
        ax.add_patch(
            patches.Rectangle(
                (x - x1, y - y1), w, h,
                linewidth=1.5, edgecolor="lime", facecolor="none"
            )
        )
        ax.set_title(f"sigma={s}, amount={a}")
        ax.axis("off")

plt.tight_layout()
plt.show()


In [27]:
# Black-hat kernel 튜닝 (어두운 혈관 강조)
# kernel은 bbox 평균 크기(대략 60px)보다 약간 작게/비슷하게 시작

kernel_sizes = [9, 15, 21, 31, 41]

fig, axes = plt.subplots(1, len(kernel_sizes) + 1, figsize=(22, 5))

crop, _, _ = crop_bbox(img, bbox, padding=40)
axes[0].imshow(crop, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

x, y, w, h = bbox

for i, k in enumerate(kernel_sizes):
    out = apply_blackhat(img, kernel_size=k)
    crop_img, x1, y1 = crop_bbox(out, bbox, padding=40)
    axes[i + 1].imshow(crop_img, cmap="gray")
    axes[i + 1].add_patch(
        patches.Rectangle(
            (x - x1, y - y1), w, h,
            linewidth=2, edgecolor="lime", facecolor="none"
        )
    )
    axes[i + 1].set_title(f"k={k}")
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()


In [28]:
# 조합 비교: 실제 파이프라인 후보
# A Safe 계열 후보도 여기서 눈으로 고르면 됨

combos = []

# 1) original
combos.append(("Original", img))

# 2) CLAHE only
c1 = apply_clahe(img, clip_limit=clip_limit, tile_grid=tile_grid)
combos.append((f"CLAHE {clip_limit}/{tile_grid}", c1))

# 3) CLAHE + Gamma
c2 = apply_gamma(c1, gamma=gamma_value)
combos.append((f"CLAHE+Gamma({gamma_value})", c2))

# 4) CLAHE + Gamma + Blackhat
c3 = apply_blackhat(c2, kernel_size=tophat_kernel)
combos.append(("CLAHE+Gamma+Blackhat", c3))

# 5) CLAHE + Frangi
c4 = apply_frangi(c1, sigmas=frangi_sigmas)
combos.append(("CLAHE+Frangi", c4))

# 6) CLAHE + Gamma + Frangi
c5 = apply_frangi(c2, sigmas=frangi_sigmas)
combos.append(("CLAHE+Gamma+Frangi", c5))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
x, y, w, h = bbox

for ax, (title, im) in zip(axes.ravel(), combos):
    crop_img, x1, y1 = crop_bbox(im, bbox, padding=40)
    ax.imshow(crop_img, cmap="gray")
    ax.add_patch(
        patches.Rectangle(
            (x - x1, y - y1), w, h,
            linewidth=2, edgecolor="lime", facecolor="none"
        )
    )
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


### 튜닝 결과 반영
위에서 고른 값을 아래 셀에서 다시 지정한 뒤, `visualize_preprocessing` / batch 저장을 재실행.


In [29]:
# ===== 최종 선택 파라미터 (여기만 수정) =====

# CLAHE
clip_limit = 2.0
tile_grid = (4, 4)   # 노트 메모 기준 후보. 그리드 튜닝 보고 확정

# Gamma
gamma_value = 0.8

# Frangi
frangi_sigmas = range(1, 5)

# Unsharp
unsharp_sigma = 2.0
unsharp_amount = 1.5

# Black-hat / Top-hat kernel
tophat_kernel = 15

print("clip_limit :", clip_limit)
print("tile_grid  :", tile_grid)
print("gamma      :", gamma_value)
print("frangi     :", list(frangi_sigmas))
print("unsharp    :", unsharp_sigma, unsharp_amount)
print("kernel     :", tophat_kernel)

# 선택값으로 다시 시각화
visualize_preprocessing(
    img,
    bbox,
    image_name,
    clip_limit=clip_limit,
    tile_grid=tile_grid,
    gamma_value=gamma_value,
    frangi_sigmas=frangi_sigmas,
    unsharp_sigma=unsharp_sigma,
    unsharp_amount=unsharp_amount,
    tophat_kernel=tophat_kernel,
)


clip_limit : 2.0
tile_grid  : (4, 4)
gamma      : 0.8
frangi     : [1, 2, 3, 4]
unsharp    : 2.0 1.5
kernel     : 15


CLAHE

- clipLimit / tileGrid는 위 1D·2D 튜닝 셀에서 같이 볼 것
- tileGrid=(4,4) 후보는 유지하되, clip×tile 그리드 보고 확정

Gamma

- gamma 변화에 따른 육안 차이는 크지 않을 수 있음

추가

- Frangi: CLAHE 이후 + sigma / black_ridges 튜닝
- Black-hat: 어두운 혈관 강조용 kernel 튜닝
- 조합 비교 셀에서 최종 파이프라인 후보 선택


## BATCH Preprocessing

Save outputs to `preprocessed/{pipeline}/{train,val}/`.


In [30]:
from tqdm import tqdm

VAL_RAW_DIR = ARCADE_DIR / 'stenosis' / 'val' / 'images'

train_image_list = sorted(RAW_DIR.glob('*.png'))
val_image_list = sorted(VAL_RAW_DIR.glob('*.png'))

print(f'Train images : {len(train_image_list)}')
print(f'Val images   : {len(val_image_list)}')
print(f'Root         : {PREPROCESS_DIR}')
for folder in folders:
    print(' -', PREPROCESS_DIR / folder / 'train')
    print(' -', PREPROCESS_DIR / folder / 'val')


Train images : 1000
Val images   : 200
Root         : <PROJECT_ROOT>/data/preprocessed
 - <PROJECT_ROOT>/data/preprocessed/01_clahe/train
 - <PROJECT_ROOT>/data/preprocessed/01_clahe/val
 - <PROJECT_ROOT>/data/preprocessed/02_gamma/train
 - <PROJECT_ROOT>/data/preprocessed/02_gamma/val
 - <PROJECT_ROOT>/data/preprocessed/03_frangi/train
 - <PROJECT_ROOT>/data/preprocessed/03_frangi/val
 - <PROJECT_ROOT>/data/preprocessed/04_clahe_gamma/train
 - <PROJECT_ROOT>/data/preprocessed/04_clahe_gamma/val
 - <PROJECT_ROOT>/data/preprocessed/05_clahe_gamma_frangi/train
 - <PROJECT_ROOT>/data/preprocessed/05_clahe_gamma_frangi/val
 - <PROJECT_ROOT>/data/preprocessed/06_gamma_clahe_frangi/train
 - <PROJECT_ROOT>/data/preprocessed/06_gamma_clahe_frangi/val
 - <PROJECT_ROOT>/data/preprocessed/07_black_hat/train
 - <PROJECT_ROOT>/data/preprocessed/07_black_hat/val


In [31]:
# Save all images by split

def save_preprocessed(image_path, folder_name, split_name, out_img):
    save_path = PREPROCESS_DIR / folder_name / split_name / image_path.name
    cv2.imwrite(str(save_path), out_img)


def run_batch(image_list, split_name):
    for image_path in tqdm(image_list, desc=f'preprocess/{split_name}'):
        img_in = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if img_in is None:
            continue

        img_clahe = apply_clahe(img_in, clip_limit=clip_limit, tile_grid=tile_grid)
        save_preprocessed(image_path, '01_clahe', split_name, img_clahe)

        img_gamma = apply_gamma(img_in, gamma=gamma_value)
        save_preprocessed(image_path, '02_gamma', split_name, img_gamma)

        img_frangi = apply_frangi(img_in, sigmas=frangi_sigmas)
        save_preprocessed(image_path, '03_frangi', split_name, img_frangi)

        img_cg = apply_gamma(img_clahe, gamma=gamma_value)
        save_preprocessed(image_path, '04_clahe_gamma', split_name, img_cg)

        img_cgf = apply_frangi(img_cg, sigmas=frangi_sigmas)
        save_preprocessed(image_path, '05_clahe_gamma_frangi', split_name, img_cgf)

        img_gcf = apply_frangi(
            apply_clahe(img_gamma, clip_limit=clip_limit, tile_grid=tile_grid),
            sigmas=frangi_sigmas,
        )
        save_preprocessed(image_path, '06_gamma_clahe_frangi', split_name, img_gcf)

        # dark-vessel emphasizing morphological response
        img_black_hat = apply_blackhat(img_in, kernel_size=tophat_kernel)
        save_preprocessed(image_path, '07_black_hat', split_name, img_black_hat)

run_batch(train_image_list, 'train')
run_batch(val_image_list, 'val')


In [32]:
# Verify counts by split

for folder in folders:
    n_train = len(list((PREPROCESS_DIR / folder / 'train').glob('*.png')))
    n_val = len(list((PREPROCESS_DIR / folder / 'val').glob('*.png')))
    print(f'{folder}: train={n_train}, val={n_val}')


01_clahe: train=1000, val=200
02_gamma: train=1000, val=200
03_frangi: train=1000, val=200
04_clahe_gamma: train=1000, val=200
05_clahe_gamma_frangi: train=1000, val=200
06_gamma_clahe_frangi: train=1000, val=200
07_black_hat: train=1000, val=200
